# ML-09 - Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/8-BitRhyon/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits my own Week-5 model the way the FlyRank research paper was audited in the
Week 6 session: two methodology questions for the paper, then the same lens on my work.

What it covers:

1. Two findings from `docs/flyrank-seo-research-march-2026.pdf` and the methodology question each one earns.
2. The Week-5 model re-run under three split designs, including a time-forward split, with the before/after.
3. A leakage audit: label-derived column, future windows, product flags, and population selection.
4. Real failure examples, including the ones that expose the label definition.
5. A claim rewrite where the evidence does not carry the original sentence.

Claims use observed, measured, directional, and decision-support language throughout.


## 1. Two paper findings + my methodology questions

Both questions are about inference, not about whether the pattern exists. The paper is explicit
that its ML appendix is exploratory and that the aggregate comparisons lead. These questions are
aimed at the recommendations that follow the findings.

### Finding #3 - Click capture by position tier

The paper reports weighted portfolio CTR of 0.423% at top 3, 0.339% on page 1, and 0.050% at deep,
an 88% drop from top 3 to deep.

**Methodology question: does the pooled curve survive at the brand level, and does the design
support the action it recommends?**

The weighting is by impressions across 57 brands, so a few high-volume brands can carry most of the
curve. The paper is careful to call these portfolio-level weighted CTRs rather than generic
benchmarks. The next step would be the same curve computed within each brand, or a mixed model with
brand as a group. Separately, position and CTR are measured in the same 90-day window, which makes
the comparison an association. The recommendation to refine page-one snippets assumes a causal lift
that this design does not test. A within-page before/after on revised versus unrevised pages at the
same position would carry the action further.

### Finding #4 - The freshness multiplier

The paper reports that 365+ day pages refreshed within 30 days show a 3.2x health boost (10.7 to
34.5) and 57x more impressions (71 to 4,039).

**Methodology question: which pages get refreshed, and what does the 57x ratio rest on?**

If editors refresh pages with proven demand or strategic value, the refreshed group differs from
the untouched group before the refresh. A matched comparison on age, prior impressions, and
position, or a within-page before/after with a comparison group, would separate the refresh from
the selection. The 57x figure also starts from a small baseline: 71 impressions is a denominator
where a handful of pages can dominate the ratio, so a median or trimmed comparison would be more
reliable. The paper already applies exactly this instinct to the 361+ freshness bucket, which it
calls too small to headline. The same treatment for the 57x ratio would make the claim stronger.

### What I take from this for my own work

Both questions have the same shape: a real pattern, a recommendation that runs slightly ahead of
the design. That is the failure mode I look for in Section 5.


## 2. My model under an honest split (before/after)

The Week-5 model already used a client-grouped split. The audit asks whether that choice mattered
and whether a grouped split is enough.

Three designs, same features and same label:

| Design | What it tests | What could go wrong |
|---|---|---|
| **Random page split** (before) | Does the model fit pages it has seen siblings of? | Sibling pages from one client share templates and traffic patterns, so the score can be memorisation. |
| **Client-grouped split** (after) | Does it work on a client it has never seen? | The honest cross-sectional question, but still the same month pair. |
| **Time-forward split** | Does a model trained on January-February work on March-April? | The deployment question: the future is not the same month as the past. |

The time-forward frame uses the same label definition one period earlier: January features into a
February label. The model trained there is tested on the March-April rows, on the same test rows as
the grouped model so the comparison is apples to apples.


In [1]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from huggingface_hub import get_token
        HF_TOKEN = get_token()
    except Exception:
        HF_TOKEN = None

assert HF_TOKEN, "No Hugging Face token. Set a Colab Secret named HF_TOKEN, or run `hf auth login` once."

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

BASE = "hf://datasets/FlyRank/internship-warehouse"
FEATURES = ["ctr_pct", "avg_pos", "march_impressions_log", "position_sd", "content_age_days"]
K = 140
SEED = 42


def month_rollup(month):
    return f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp,
               SUM(gsc_clicks) AS clk,
               SUM(gsc_sum_position) AS pos_sum,
               SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions END) AS pos_imp,
               COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_gsc,
               STDDEV_SAMP(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_sd
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={month}/data_0.parquet')
        GROUP BY 1, 2
    )
    SELECT client_hash_id, content_hash_id, imp, clk, days_gsc, pos_sd,
           pos_sum / NULLIF(pos_imp, 0) AS avg_pos,
           100.0 * clk / NULLIF(imp, 0) AS ctr_pct
    FROM daily
    ORDER BY client_hash_id, content_hash_id
    """


dim = con.execute(
    f"SELECT client_hash_id, content_hash_id, content_created_date "
    f"FROM read_parquet('{BASE}/dim_content.parquet')"
).df()


def build_frame(feature_month, label_month, as_of):
    """One page-month row per page, with the next-month tier-relative label."""
    feats = con.execute(month_rollup(feature_month)).df()
    nxt = con.execute(month_rollup(label_month)).df()

    f = feats[(feats.imp >= 500) & (feats.days_gsc >= 20) & feats.avg_pos.notna()].copy()
    f["tier"] = pd.cut(
        f.avg_pos, [0, 3, 10, 20, np.inf], labels=["top_3", "page_1", "page_2", "deep"]
    ).astype(str)
    thresholds = f.groupby("tier").ctr_pct.median()

    nxt = nxt.rename(columns={"imp": "next_imp", "ctr_pct": "next_ctr"})
    f = f.merge(nxt[["client_hash_id", "content_hash_id", "next_imp", "next_ctr"]],
                on=["client_hash_id", "content_hash_id"], how="inner")
    f = f[(f.next_imp >= 100) & f.next_ctr.notna()].copy()
    f["label"] = (f.next_ctr < f.tier.map(thresholds)).astype(int)

    f = f.merge(dim, on=["client_hash_id", "content_hash_id"], how="left")
    f["march_impressions_log"] = np.log1p(f.imp)
    f["position_sd"] = f.pos_sd.fillna(0)
    f["content_age_days"] = (pd.Timestamp(as_of) - pd.to_datetime(f.content_created_date)).dt.days
    f["content_age_days"] = f.content_age_days.fillna(f.content_age_days.median())

    return f.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)


early = build_frame("2026-01", "2026-02", "2026-01-31")
late = build_frame("2026-03", "2026-04", "2026-03-31")

print(f"early frame (Jan into Feb): {len(early):,} pages, {early.client_hash_id.nunique()} clients, "
      f"base rate {early.label.mean():.4f}")
print(f"late frame  (Mar into Apr): {len(late):,} pages, {late.client_hash_id.nunique()} clients, "
      f"base rate {late.label.mean():.4f}")
print(f"clients present in both periods: {len(set(early.client_hash_id) & set(late.client_hash_id))}")


early frame (Jan into Feb): 34,791 pages, 25 clients, base rate 0.4716
late frame  (Mar into Apr): 58,477 pages, 31 clients, base rate 0.5719
clients present in both periods: 23


In [2]:
X = late[FEATURES].fillna(0)
y = late.label
groups = late.client_hash_id


def precision_at_k(scores, labels, k=K):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())


def fit_model(train_frame, train_idx=None):
    model = HistGradientBoostingClassifier(max_iter=200, random_state=SEED)
    if train_idx is None:
        model.fit(train_frame[FEATURES].fillna(0), train_frame.label)
    else:
        model.fit(train_frame[FEATURES].fillna(0).iloc[train_idx], train_frame.label.iloc[train_idx])
    return model


# The shared test rows: the 30% client-grouped holdout.
grouped_train, grouped_test = next(
    GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED).split(X, y, groups=groups)
)
test_labels = y.iloc[grouped_test].to_numpy()

# BEFORE: random page split, clients shared across the fence.
random_train, random_test = train_test_split(np.arange(len(late)), test_size=0.30, random_state=SEED)
random_model = fit_model(late, random_train)
random_scores = random_model.predict_proba(X.iloc[random_test])[:, 1]

# AFTER: client-grouped split.
grouped_model = fit_model(late, grouped_train)
grouped_scores = grouped_model.predict_proba(X.iloc[grouped_test])[:, 1]

# TIME-FORWARD: trained one period earlier, tested on the same grouped holdout rows.
time_model = fit_model(early)
time_scores = time_model.predict_proba(X.iloc[grouped_test])[:, 1]

rows = [
    {
        "design": "random page split (before)",
        "train_pages": len(random_train),
        "test_pages": len(random_test),
        "train_test_client_overlap": len(set(groups.iloc[random_train]) & set(groups.iloc[random_test])),
        "AUC": roc_auc_score(y.iloc[random_test], random_scores),
        "precision_140": precision_at_k(random_scores, y.iloc[random_test]),
    },
    {
        "design": "client-grouped (after)",
        "train_pages": len(grouped_train),
        "test_pages": len(grouped_test),
        "train_test_client_overlap": len(set(groups.iloc[grouped_train]) & set(groups.iloc[grouped_test])),
        "AUC": roc_auc_score(test_labels, grouped_scores),
        "precision_140": precision_at_k(grouped_scores, test_labels),
    },
    {
        "design": "time-forward Jan-Feb into Mar-Apr",
        "train_pages": len(early),
        "test_pages": len(grouped_test),
        "train_test_client_overlap": len(set(early.client_hash_id) & set(groups.iloc[grouped_test])),
        "AUC": roc_auc_score(test_labels, time_scores),
        "precision_140": precision_at_k(time_scores, test_labels),
    },
]

comparison = pd.DataFrame(rows)
print(f"test base rate: {test_labels.mean():.4f} | K = {K}")
comparison.round(4)


test base rate: 0.5669 | K = 140


,design,train_pages,test_pages,train_test_client_overlap,AUC,precision_140
0,random page split (before),40933,17544,30,0.8356,0.9857
1,client-grouped (after),27231,31246,0,0.8308,0.9857
2,time-forward Jan-Feb into Mar-Apr,34791,31246,9,0.7875,0.8857


**Before/after, read honestly.** `train_test_client_overlap` counts clients on both sides of the
split. The time-forward row still shares clients with the test period by design: the model knows
each client's earlier history. It tests temporal generalization, not client generalization.

- **Random to grouped does not move the number much.** AUC 0.8356 to 0.8308, precision@140
  unchanged at 0.9857, while 30 of 31 clients appear on both sides of the random fence. The reason is
  that the five features are page-level and generic, so there is little client-specific structure to
  memorise. Grouping is still the correct design. Here it guards against a risk that happens not to
  bite. That gap is itself a finding: this model was not secretly reading the client.
- **The time-forward split does move it.** Trained on January-February and tested on the same
  March-April holdout rows, AUC falls from 0.8308 to 0.7875 and precision@140 falls from 0.9857 to
  0.8857. About 14 of the 140 slots are lost when the model has to predict a later period.
- **The rule on the same rows** reaches precision@140 0.9786. Under the time-forward test the rule
  beats the model by 13 pages. The model's same-period edge does not survive the deployment-like
  test.

The 0.986 number from ML-08 is therefore a same-period estimate. The defensible forward-looking
number is closer to 0.886, and at that level the rule is still the better queue.


## 3. Leakage audit

Four checks, in the order the leakage skill lists them.

**Label-derived features.** The label is built from April CTR. Adding that column back as a feature
should push the score toward perfect. It does.

**Future windows.** Every feature comes from March 2026. No April column may enter the feature list.

**Product flags.** The warehouse frame has no FlyRank flags, and the starter-file flags
(`trend_direction`, `is_declining_label`) are not in this frame. Nothing decision-derived is used.

**Population selection.** The label requires at least 100 April impressions. That is outcome-month
information deciding who enters the evaluated cohort. It is disclosed, not hidden: 829 of 59,306
eligible pages (1.4%) are dropped by it.


In [3]:
leak_features = FEATURES + ["april_ctr_pct"]
late["april_ctr_pct"] = late.next_ctr

leak_model = HistGradientBoostingClassifier(max_iter=200, random_state=SEED)
leak_model.fit(late[leak_features].fillna(0).iloc[grouped_train], y.iloc[grouped_train])
leak_scores = leak_model.predict_proba(late[leak_features].fillna(0).iloc[grouped_test])[:, 1]

print(f"AUC with the label source added: {roc_auc_score(test_labels, leak_scores):.4f}")
print(f"precision@140 with the label source added: {precision_at_k(leak_scores, test_labels):.4f}")

late = late.drop(columns=["april_ctr_pct"])
assert "april_ctr_pct" not in late.columns
assert not any(c.startswith("april") or c.startswith("next_") for c in FEATURES)

assert "april_ctr_pct" not in FEATURES
print("\nfeatures used:", FEATURES)

forbidden = {"trend_direction", "trend_pct", "is_declining_label", "health_score", "health"}
print("product flags present in the feature list:", sorted(forbidden & set(FEATURES)))

print(f"\nrows in the evaluated late frame: {len(late):,}")
print("pages dropped by the April 100-impression floor: 829 of 59,306 eligible (1.4%), disclosed in ML-04")


AUC with the label source added: 1.0000
precision@140 with the label source added: 1.0000

features used: ['ctr_pct', 'avg_pos', 'march_impressions_log', 'position_sd', 'content_age_days']
product flags present in the feature list: []

rows in the evaluated late frame: 58,477
pages dropped by the April 100-impression floor: 829 of 59,306 eligible (1.4%), disclosed in ML-04


**Leakage audit result.** The label-derived column takes AUC from 0.8308 to 1.0000 and
precision@140 to 1.0000. Removing it restores the honest number, and the assertions confirm no April
column survives in the feature list. No product flag is present. The population floor is the one
place where the outcome month influences the cohort, and it is disclosed rather than hidden.

**Feature sanity check.** The top feature is March CTR, with a permutation AUC drop of 0.313 in
ML-08, followed by position at 0.057. That ordering is plausible for a CTR label. Nothing towers at
a near-perfect level, which is what the leak test above looks like when a real leak is present.


## 4. Failure examples

Two directions, because a queue can fail by spending a slot on a bad page or by missing a good one.

**False positives.** The model put these pages in the top 140, and April disagreed.

**False negatives.** These pages are in the bottom of the ranking and the label says positive. They
are the more interesting failures, because they expose the label definition rather than the model.


In [4]:
audit = late.iloc[grouped_test].copy()
audit["y_true"] = test_labels
audit["score"] = grouped_scores
audit["rank"] = audit.score.rank(ascending=False, method="first").astype(int)

false_positives = audit[(audit["rank"] <= K) & (audit.y_true == 0)].sort_values("rank")
false_negatives = audit[(audit["rank"] > K) & (audit.y_true == 1)].sort_values("score")

cols = ["rank", "tier", "imp", "ctr_pct", "avg_pos", "next_ctr", "content_age_days"]
print("false positives inside the top 140:")
print(false_positives[cols].round(3).to_string(index=False))
print("\nlowest-scored positives (false negatives):")
print(false_negatives[cols].head(3).round(3).to_string(index=False))

print("\nlabel share among 'deep' pages in this test set:",
      round(audit[audit.tier == "deep"].y_true.mean(), 3))
print("median March CTR of 'deep' pages:",
      round(audit[audit.tier == "deep"].ctr_pct.median(), 3))


false positives inside the top 140:
 rank  tier    imp  ctr_pct  avg_pos  next_ctr  content_age_days
   40 top_3 2826.0    0.035    2.632     0.679                95
  105 top_3 5605.0    0.018    0.981     0.345                54

lowest-scored positives (false negatives):
 rank   tier     imp  ctr_pct  avg_pos  next_ctr  content_age_days
31142   deep 12359.0    0.550   23.583     0.000               153
31082 page_1  1587.0    2.205    9.646     0.000               230
31035   deep  8421.0    0.594   24.444     0.064               166

label share among 'deep' pages in this test set: 0.546
median March CTR of 'deep' pages: 0.078


**False positives.** Both are `top_3` pages with a March CTR near zero (0.035% and 0.018%) whose
April CTR recovered to 0.679% and 0.345%. The model read a large gap against the tier median and
expected the gap to persist. The month was the anomaly. A top-three page with almost no clicks for a
whole month is rare enough that the honest action is to check the query report before rewriting.

**False negatives.** The clearest case is a `deep` page with 12,359 impressions and a 0.55% March
CTR that fell to 0.000 in April. The model ranked it 31,142 of 31,246. On March evidence that page
looked strong: the `deep` tier median is about 0.074%, so 0.55% is far above it, and the model had
no reason to expect a collapse. The label is tier-relative, so a page that was strong in absolute
terms becomes a positive when it falls below a low tier median. That is a property of the label
definition, not a model bug, and it belongs in the limitations.

A zero April CTR can also be a tracking gap. The label treats it as a positive, so the cohort
contains some pages whose outcome is a measurement failure rather than a content signal. That is
worth a manual check before the queue is used.


## 5. Claim rewrite

Five sentences from earlier notebooks, checked against what the evidence now supports.

| Original claim | Where | What the evidence supports | Rewritten claim |
|---|---|---|---|
| "precision@140 at least 0.80 and 15 points above the baseline" | ML-03 | The rule baseline reaches 0.979 on the same-period split, so a 15-point margin is impossible against a 1.000 ceiling. | "The model must beat the rule at the same K on unseen clients. The bar is relative, not absolute." |
| "The model's AUC gain is real and consistent across most clients" | ML-08 | Same-period AUC 0.831 vs 0.763, better on 6 of 7 substantial clients. Time-forward precision@140 falls to 0.886 against the rule's 0.979. | "Measured AUC is higher on a same-period grouped split. The advantage does not survive a time-forward split, so it is not a deployment claim." |
| "At K=140 the two methods are tied, so the rule ships" | ML-08 | Same-period tie (3 of 5 seeds). Time-forward the rule wins by 13 pages. | "At K=140 the rule ties the model on the same-period split and wins under the time-forward split. Decision-support: the rule is the default." |
| "Revisit the model if the budget grows toward 5,000 pages" | ML-08 | A single-split observation that has not been tested time-forward. | "The K=5000 advantage is a single-split observation. Treat it as a hypothesis for the next iteration, not a recommendation." |
| "The gap is worth 653 clicks" | ML-07 | The score assumes the page reaches its tier median. It is an upper bound used for ranking. | "The score estimates the click gap if the page reached its tier median. It is decision-support for ordering, not a forecast of recovered clicks." |

The pattern in every rewrite: the original sentence was true inside one split and one definition,
and the rewrite names the condition instead of dropping it.


In [5]:
summary = pd.DataFrame([
    {"design": "random page split", "AUC": roc_auc_score(y.iloc[random_test], random_scores),
     "precision_140": precision_at_k(random_scores, y.iloc[random_test])},
    {"design": "client-grouped", "AUC": roc_auc_score(test_labels, grouped_scores),
     "precision_140": precision_at_k(grouped_scores, test_labels)},
    {"design": "time-forward", "AUC": roc_auc_score(test_labels, time_scores),
     "precision_140": precision_at_k(time_scores, test_labels)},
])
print("final audit numbers, same features and same label definition:")
print(summary.round(4).to_string(index=False))
print(f"\nlabel-derived leak check: AUC {roc_auc_score(test_labels, leak_scores):.4f} before removal")
print("honest numbers restored:", round(comparison.loc[1, 'AUC'], 4), "AUC and",
      round(comparison.loc[1, 'precision_140'], 4), "precision@140 on the grouped split")


final audit numbers, same features and same label definition:
           design    AUC  precision_140
random page split 0.8356         0.9857
   client-grouped 0.8308         0.9857
     time-forward 0.7875         0.8857

label-derived leak check: AUC 1.0000 before removal
honest numbers restored: 0.8308 AUC and 0.9857 precision@140 on the grouped split


## Self-check

- [x] Two paper findings named, each with a constructive methodology question about labels or design
- [x] The model re-run under a random split, a client-grouped split, and a time-forward split
- [x] Before/after shown: grouped changes little, time-forward costs about 10 pages at K=140
- [x] Leakage audit: label-derived column tested and removed, future windows checked, flags checked
- [x] Population selection disclosed: 829 pages (1.4%) dropped by the April floor
- [x] Real failure examples in both directions, with the label-definition issue named
- [x] Claims rewritten where the evidence is narrower than the original sentence
- [x] Safe claim language throughout: observed, measured, directional, decision-support
- [x] The notebook ran top to bottom with no errors (executed before commit)
- [x] No client names, URLs, private queries, or tokens anywhere
- [x] Committed under `work/notebooks/`
